# Part 6: Transformer Vision Models

从 patch token 与 self-attention 理解 ViT、Swin 和 SegFormer，并关注遥感大图计算复杂度。

**先修要求**：完成前序章节，了解 NumPy、张量 shape 与训练/验证划分。

## 学习目标与路线

1. 明确问题与输入输出
2. 理解核心结构
3. 运行最小实验
4. 检查指标和 shape
5. 完成练习与遥感迁移

`image → patch embedding → tokens + position → attention/MLP blocks → multi-scale decoder`

In [ ]:
from pathlib import Path
import json
ROOT = Path.cwd().resolve()
while not (ROOT / 'scripts').exists() and ROOT.parent != ROOT: ROOT = ROOT.parent
print('repository:', ROOT)

## 核心概念

- **Patch embedding**：把图像切块并投影为 token 序列。
- **Self-attention**：用 QKᵀ/√d 计算 token 间关系，再加权 V。
- **Hierarchical features**：Swin/SegFormer 逐级降采样，更适合密集预测。
- **Complexity**：全局 attention 随 token 数平方增长，大图需窗口、稀疏或多尺度策略。

## 本地真实训练结果

长训练由脚本执行；这里读取本机已经生成的指标和产物。GitHub 中不会包含 `outputs/`。

In [ ]:
output = ROOT / r'outputs/part6_segformer_b0'
metrics_path = output / 'metrics.json'
print('local output:', output)
print(json.dumps(json.loads(metrics_path.read_text(encoding='utf-8')), indent=2) if metrics_path.exists() else '尚未运行：请执行下方命令。')

## 可恢复的脚本训练

默认不在 Notebook 内启动长训练。将 `RUN_SHORT_TRAIN` 改为 `True` 可执行一次小规模 smoke run；正式命令请在终端执行。

In [ ]:
RUN_SHORT_TRAIN = False
command = r'''conda run --no-capture-output -n remote-sensing-dl python scripts/real_training/train_segmentation.py --model segformer_b0 --data-root data/landcoverai --output outputs/part6_segformer_b0 --epochs 1 --train-limit 32 --val-limit 16'''
print(command)
if RUN_SHORT_TRAIN:
    import subprocess
    subprocess.run(command, cwd=ROOT, shell=True, check=True)

## 结果解释

正式结果必须包含非空 checkpoint、完整 epoch 日志、验证指标与预测图；出现缺失时先检查数据配对、CUDA 与配置，而不要直接扩大模型。

## 练习

分别计算 512×512 影像在 patch size 16 与 8 下的 token 数及 attention 矩阵元素数量。

In [ ]:
# 答案脚手架
experiment = {'hypothesis': '', 'controlled_variables': [], 'metric': '', 'expected_failure': ''}
print(experiment)

## 常见坑与扩展

**常见坑**：位置编码尺寸不匹配；忽略多光谱通道适配；token 太多导致显存平方增长；只看自然图像预训练而不做域验证。

**扩展**：替换为真实遥感数据时，记录传感器、波段、分辨率、空间划分和随机种子。

## 完成检查

- [ ] 所有单元格从头运行
- [ ] 指标实现通过 sanity check
- [ ] 没有 train/val 空间泄漏
- [ ] 配置和随机种子已记录
- [ ] 至少分析一个失败案例